### BART
- transformer 기반의 모델
    - encoder : 문장을 이해
    - decoder : 문장을 생성
- Encoder, Decoder 혼합 모델
- 번역, 요약, 오타를 찾아서 새로운 텍스트 구성
- Encoder : BERT 모델의 인코더 방식을 사용하여 문장을 이해
- Decoder : 출력이 되는 문장은 GPT 방식으로 생성
- 평가 지표를 확인하는 방법은 n-gram을 이용하여 같은 단어를 사용했는가? -> 로그 스케일 수치 값을 출력
- input tokenizer와 output의 tokenizer를 따로 사용
- transformer 모델들은  tokenizer를 sentencepiece 사용

In [1]:
# !pip install lighteval
# !pip install rouge_score evaluate

In [2]:
#문장 간의 검증 지표를 만들어주는 라이브러리
import evaluate
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#특수 토큰
#<PAD> : 빈칸 채우기
#<UNK> : OOV
#<SEP> : 2번쨰 문장
#<EOS> : 전체 문장의 끝, 예측값이 언제 끝나는가? <\s>
model_name='gogamza/kobart-summarization'

In [4]:
#tokenizer, model 생성
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 1236.28it/s]


In [5]:
train_docs=[
    '정부는 중소기업 세제 혜택과 R&D 세액 공제를 확대한다고 밝혔다',
    '해당 기업은 분기 실적에서 매출 성장을 기록했으며 신제품 출시를 예고했다'
]
train_sums=[
    '정부가 중소기업 지원을 확대한다',
    '기업이 실적 개선과 신제품 출시를 예고했다'
]

valid_docs=[
    '교육부가 디지털 교과서 도입을 추진한다고 밝혔다'
]
valid_sums=[
    '교육부가 디지텉 교과서 ㄷ입을 추진한다.'
]

In [6]:
#transformer 모델에서 사용하는 데이터
#DatasetDict는 Dataset을 한번에 
raw_ds=DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document' : train_docs,
                'summary' : train_sums,
            }
        ),
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs,
                'summary' : valid_sums,
            }
        )
        
    }
)
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 1
    })
})

In [7]:
#입력 / 출력 문장의 최대 길이를 설정
max_input_len=512
max_target_len=128

In [8]:
#tokenizer 함수 
def token_fn(batch):
    #batch : 배치로 묶인 데이터
    #inputs -> 독립 변수 
    inputs = tokenizer(
        batch['document'], 
        max_length = max_input_len, 
        padding = 'max_length',         #고정 길이의 벡터를 사용
        trunction = True                #최대 길이보다 큰 경우 자른다.
    )
    #출력 데이터 인코딩
    labels = tokenizer(
        batch['summary'], 
        max_length = max_target_len, 
        padding = True, 
        truncation = True
    )

    #padding토큰의 인덱스 값은 일반적으로 0
    #labels의 padding토큰의 인덱스 값을 -100으로 변환 
    #-100으로 변환하는 이유는 -> CrossEntropyLoss()에서 -100은 무시 할수 있는 차원으로 구성 
    #tokenizer의 결과 -> attention_mask(실제 토큰, 패딩 토큰), input_ids(인코딩된 단어들)
    labels_ids = np.array( labels['input_ids'] )
    labels_ids[ labels_ids  == tokenizer.pad_token_id ] = -100
    #labels의 데이터 -> 정답 
    #inputs에 labels 새로운 키를 생성하여 데이터를 대입 
    inputs['labels'] = labels_ids.tolist()

    return inputs

In [9]:
# raw_ds 데이터를 token_fn에 대입 
tokenized_ds = raw_ds.map(
    token_fn, 
    batched=True, 
    remove_columns= ['document', 'summary']
)

tokenized_ds

Map: 100%|██████████| 1/1 [00:00<00:00, 45.16 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})

In [10]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer= tokenizer, 
    model = model
)

In [11]:
#검증 지표 선택 
rouge = evaluate.load('rouge')
#BART모델은 생성된 문장과 정답 문장 사이에 얼마나 많은 단어가 공통으로 등장하였는가? 비율을 계산

#3개의 연산
#ROUGE-1 : 개별 단어(1-gram)가 얼마나 겹치는가?
#ROUGE-2 : 연속된 2단어(2-gram)가 얼마나가 겹치는가?
#ROUGE-L : 가장 길게 공통으로 이어지는 문자열을 기반으로 측정 

In [12]:
#생성된 문장과 정답 문장을 일반적인 토큰화 작업이 필요
from konlpy.tag import Komoran
komoran=Komoran()

In [13]:
#검증 함수 
def metrics(eval_pred):
    #eval_pred : 예측값, 실젯값
    preds, labels = eval_pred

    #padding token의 인덱스의 값으로 labels의 -100의 값들을 재 변경 
    #decode() 함수를 이용해서 인코딩된 단어들을 다시 단어로 변환 
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )

    #텍스트로 디코딩 작업 
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    #문장에서 좌우의 공백이 존재하는 경우에는 다른 값으로 측정하기 때문에 각 문장 별로 좌우의 공백을 제거 
    pred_str = [doc.strip() for doc in pred_str]
    label_str = [doc.strip() for doc in label_str]

    #ROUGE 계산식 
    result = rouge.compute(
        predictions= pred_str, 
        references= label_str, 
        tokenizer = lambda x : komoran.morphs(x) 
    )

    result = { k : round(v * 100, 2) for k, v in result.items() }

    return result

In [14]:
args = Seq2SeqTrainingArguments(
    output_dir= "./kobart", 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    learning_rate= 5e-05,
    num_train_epochs=5, 
    logging_steps=2, 

    load_best_model_at_end= True, 
    metric_for_best_model= 'rougeL', 
    greater_is_better=True, 

    #generate 설정을 변경 
    #평가 시 직접 문장을 생성할것인가?
    predict_with_generate=True, 
    #생성할 문장의 최대 토큰의 길이
    generation_max_length= 64, 
    #데이터 생성 시 문장 후보의 탐색의 개수 
    generation_num_beams= 4
)

In [15]:
#Trainer 생성 
trainer = Seq2SeqTrainer(
    model = model, 
    args = args, 
    train_dataset = tokenized_ds['train'], 
    eval_dataset= tokenized_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= data_collator, 
    compute_metrics= metrics
)

trainer.train()

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,6.659057,0.000000,0.000000,0.000000,0.000000
2,8.339540,6.806679,16.670000,5.710000,16.670000,16.670000
3,8.339540,6.952646,0.000000,0.000000,0.000000,0.000000
4,5.993108,6.667709,0.000000,0.000000,0.000000,0.000000
5,5.993108,6.175956,0.000000,0.000000,0.000000,0.000000


Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.80s/it]
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:10<00:00, 10.32s/it]
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:11<00:00, 11.49s/it]
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████

TrainOutput(global_step=5, training_loss=6.800743961334229, metrics={'train_runtime': 360.524, 'train_samples_per_second': 0.028, 'train_steps_per_second': 0.014, 'total_flos': 3048682291200.0, 'train_loss': 6.800743961334229, 'epoch': 5.0})

In [17]:
test_text = """과학기술정보통신부는 초거대 AI 연구 인프라 지원을 강화한다고 밝혔다. 
스타트업 대상으로 GPU 리소스를 확대 제공할 계획이다."""

inputs = tokenizer(
    test_text, 
    return_tensors = 'pt', 
    truction = True, 
    max_length = max_input_len
)
inputs

{'input_ids': tensor([[  0, 414, 417, 473, 414, 370, 416, 480, 416, 480, 473, 415, 417, 416,
         468, 480, 415, 372, 415, 461, 416, 414, 371, 370, 415, 469, 461, 264,
         272, 461, 416, 475, 370, 414, 367, 461, 416, 417, 464, 415, 461, 416,
         364, 416, 477, 416, 464, 461, 414, 370, 473, 417, 417, 473, 478, 415,
         468, 361, 414, 480, 461, 415, 370, 417, 476, 415, 468, 361, 245, 461,
         445, 416, 361, 417, 463, 417, 416, 475, 465, 461, 415, 469, 416, 463,
         416, 478, 415, 358, 478, 461, 270, 279, 284, 461, 415, 363, 367, 416,
         469, 416, 361, 415, 362, 461, 417, 473, 415, 469, 461, 416, 480, 478,
         414, 417, 473, 480, 461, 414, 464, 417, 416, 415, 468, 361, 245,   1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1,

In [18]:
generate_ids = model.generate(
    **inputs, 
    #출력 토근의 최대 길이 
    max_new_tokens = 64, 
    #출력 토큰의 최소 길이 
    min_new_tokens = 5, 
    #n개의 후보군 문장을 생성하여 가장 가능성이 높은 문장을 선택 
    num_beams = 4,
    #샘플링을 사용할 것인가? 빔 서치를 사용하게 되면 False / True는 확률에 따라 랜덤 생성
    do_sample = False, 
    
    length_penalty = 0.6,   #num_beams 중에 문장이 길어질때 점수를 많이 줄것인가? 깍을 것인가 지정 
                            #기본값은 1 / 
                            #1보다 작게 설정 : 짧게 쓸수록 가산점, 컴펙트한 요약
                            #1보다 크게 설정 : 길게 쓸수록 가산점, 문장을 더 길고 풍부하게 생성
    #반복 방지 -> 단어 배치가 똑같은 단어 뮦음의 개수
    no_repeat_ngram_size = 3, 
    #토큰 반복 패턴이 나타나는 경우 패널티 적용
    repetition_penalty = 1.1, 
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=5) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
print(tokenizer.decode(generate_ids[0], skip_special_tokens=True))

�������질랜드질랜드���시스코����▁공감언론▁공감언론▁공감언론������▁신청할����ạ����▁공감언론��숨을�����케이션케이션케이션찰�시스코시스코�케이션��


In [20]:
len(generate_ids[0])

65